## ELEX 4653 Lab 2

_Version 2: corrected marking code for Q4._

This lab provides practice on module imports, iterators, exception, and dictionaries.

Instructions:

- Modify this notebook by adding the Python code described below.

- Test your code using the menu item `Cell  ► Run All` or `Run ► Run All Cells`

- Save the notebook (the .ipynb file) and upload it to the appropriate Assignment folder on the course web site.

#### Question 1

Define a function `foolsday(y)` that imports the `datetime` module and returns the weekday of April 1 on the year `y` (as a string).

For example, `foolsday(2026)` should return `'Wednesday'`.  _Hint: `date.strftime()`_

In [1]:
def foolsday(y):
    import datetime
    d = datetime.date(y, 4, 1)
    return d.strftime('%A')

print(foolsday(2026))

Wednesday


#### Question 2

Define a function `unpair(l)`
that is passed a list of two-element tuples `l`.  The function should return a list of two tuples where the first tuple contains all of the first items from each tuple and the second contains all of the second items from each tuple.  Your function _must_ use the `zip` iterator.

For example, `unpair([(1,2),('a','b'),('n',100)])` would return `[(1, 'a', 'n'), (2, 'b', 100)]`

In [2]:
def unpair(l):
    return list(zip(*l))

print(unpair([(1,2),('a','b'),('n',100)]))

[(1, 'a', 'n'), (2, 'b', 100)]


#### Question 3

Declare a function named `excheck(f,l)` that calls the function `f` with each successive element from `l` and returns a list of the values in `l` that did not raise an exception.

For example, if the function `f` raised an exception for even numbers, the `excheck(f,[0,1,2,3,4])` would return `[1,3]`.

In [7]:
def excheck(f, l):
    result = []
    for x in l:
        try:
            f(x)
            result.append(x)
        except:
            pass
    return result

def f(x):
    if x % 2 == 0:
        raise ValueError
        
print(excheck(f, [0,1,2,3,4]))

[1, 3]


#### Question 4

Write a function `checkdict(d)` that takes a dictionary `d` and raises a ValueError exception if any of the values in `d` are false (e.g. 0 or empty). The exception should include a value equal to the key of the item that caused the exception.

For example, `checkdict({1:2,3:"",5:6})` would cause a ValueError exception with value `3` while  `checkdict({1:2,3:4,5:6})` would not raise an exception.

In [4]:
def checkdict(d):
    for key, value in d.items():
        if not value:
            raise ValueError(key)

try:
    checkdict({1:2, 3:"", 5:6})
except ValueError as e:
    print(e)


3


#### Question 5

Write a function `makedict(l)` that takes a list of two-element tuples `l` and returns a dict whose keys are the first elements in each tuple and whose values are tuples containing all the second elements of the tuples with that first element.

For example, `makedict([('a',1),('b',2),('a',"hi"),('a'," there"),(4,5),('b',3)])` would return `{'a': (1, 'hi', ' there'), 'b': (2, 3), 4: (5,)}`.

In [5]:
def makedict(l):
    result = {}
    for key, value in l:
        if key in result:
            result[key] += (value,)
        else:
            result[key] = (value,)
    return result

print(makedict([('a',1),('b',2),('a',"hi"),('a'," there"),(4,5),('b',3)]))

{'a': (1, 'hi', ' there'), 'b': (2, 3), 4: (5,)}


In [6]:
# lab validation code; do not modify
def labcheck(testing=0,ntest=10):
    '''
    Python exercise checking.
    Ed.Casas 2023-5-22
    Calls functions q<n>* and checks HMAC of return value[0].
    On mismatch prints return value[1] (function, arguments and return values).
    Setting testing=1 prints HMACs of correct results; paste into 'hashvalues'.
    Note:
    If q<n>* result not JSON-able, convert to string.
    Result order matters for comparison. Sort result if ordering not important.
    '''
    
    import base64, copy, hashlib, json, random, re, string, types 
    from random import randint, uniform
    import traceback
    
    # compare regex to strings that should/shouldn't match
    def checkre(pat,ok,nok):
        for s in ok:
            assert re.search(pat,s), \
                f"pattern '{pat}'\n did not match string '{s}'"
        for s in nok:
            assert not re.search(pat,s), \
                f"pattern '{pat}'\n matched string '{s}'"  
    
    # list of n words with nl letters from chars without repeats
    def randwords(n,chars=string.ascii_lowercase,nl=(2,5)):
        l = []
        while len(l)<n:
            w = ''.join([chars[randint(0,len(chars)-1)] for i in range(randint(*nl))])
            if w not in l:
                l.append(w)
        return l
    
    # convert sets to dicts and dict keys to strings so they can be sorted
    def orderkeys(o):
        if isinstance(o,set):
            return {str(k):None for k in o}
        if isinstance(o,dict):
            return {str(k):orderkeys(v) for k,v in o.items()}
        return o

    import ast, inspect
    def uses(f,s):
        for t in ast.walk(ast.parse(inspect.getsource(f))):
            if isinstance(t, ast.Call):
                if isinstance(t.func, ast.Name) and t.func.id == s:
                    return True    
                    
    import math
    def roundn(num, n):
        return 0 if not num else round(num, -int(math.floor(math.log10(abs(num)))) + (n - 1))

    def ch(s,n=(1,1)):
        return randwords(1,chars=s,nl=n)[0]

    def Q(f,a,*,rounding=None):
        oa = copy.deepcopy(a)
        r = f(*a)
        r = roundn(r,rounding) if rounding else r
        return (r,f"{f.__name__}({repr(oa[0].__name__)},{repr(oa[1])}) returns {repr(r)}")    

    def Q1():
        f,a = foolsday,(randint(1900,2026),)
        oa = copy.deepcopy(a)
        r = repr(f(*a))
        return r,f"{f.__name__}({repr(oa[0])}) returns {r}"   
        
    def Q2():
        if not uses(unpair,'zip'):
            return ([],"unpair() does not use the zip() iterator.")
        n = randint(5,7)
        f,a = unpair,(list(zip(randwords(n),[randint(1,10) for i in range(n)])),)
        oa = copy.deepcopy(a)
        r = f(*a)
        return r,f"{f.__name__}{oa}[0] returns {repr(r)}"
    
    def Q3():
        def f(x):
            i = randint(0,2)
            if i == 1:
                raise ValueError
            elif i == 2:
                raise Exception
            else:
                pass
        return Q(excheck,(f,randwords(randint(2,5))))

    def Q4():
        n=randint(4,8)
        f = checkdict
        a = dict(zip(randwords(n,nl=[6,10]), list(range(10)) ))
        oa = copy.deepcopy(a)
        try:
            f(a)
            r = "raised no exception"
        except ValueError as e:
            r = f"raised ValueError exception with string '{e}'"
        except Exception as e:
            r = f"raised wrong exception: {e}"
        return (r,f"{f.__name__}({repr(oa)}) {r}")
        
    def Q5():
        l=5*["dog","carrot","rock","cat","chair"]
        random.shuffle(l)
        l = list(zip(l[0:5],[randint(0,10) for i in range(5)]))
        f,a = makedict,l
        oa = copy.deepcopy(a)
        r = f(a)
        return r,f"{f.__name__}({repr(oa)}) returns {repr(r)}"

    hashvalues = '''
1HfP1HfPMKWw6h3zMKWw1HfP6h3zMKWwMKWwgP8L
wZ5NBAk+LMmvr4h57UpO6WBDW5vYbp9SXGPN6wxm
RACWFM0/SgZOq1ESelBpDKPb8mfHvWUn78GBQuPU
QHg1dmpC3dsQ/Wilblig2aynQR43L5KbAze38uXo
brI58aZBL8LRphv34oiod278EpA6KmqL+0nZDD2d
'''.split()

    newhash = ''
    dsize = 3 # HMAC base64 digest size (bytes, use 3 or 6 for 4 or 8 char digests)
    dlen = ((dsize*8+5)//6+3)//4*4
           
    for n,f in [(n,f) for n,f in locals().items() if callable(f) and re.search(r'^[Qq]\d+.*',n)]:
        random.seed(n)      
        hashes = '0'*dlen*ntest if testing else hashvalues.pop(0)
        err = ''
        while hashes and not err:
            h, hashes = hashes[:dlen], hashes[dlen:] 
            try:
                v,s = f()
                b = json.dumps(orderkeys(v),sort_keys=True).encode()
                c = base64.b64encode(hashlib.blake2b(b,digest_size=dsize).digest()).decode()
                if testing:
                    print(s)
                    newhash += c
                else:
                    if c != h:
                        err = f"Wrong result for test {n}: {s} (HMAC={c} instead of {h})"
            except Exception as e:
                traceback.print_exc()
                err = f"Error during test {n}: {e}"               
        if testing:
           newhash += '\n'
        else:
            print(err or f"Passed test {n}.")
            
    if testing:
        print(newhash)


labcheck(0)

Passed test Q1.
Passed test Q2.
Passed test Q3.
Passed test Q4.
Passed test Q5.
